# Minimum selected group count experiment

Test whether requiring more selected people from each group reduces LDA-exp's disparity
when allocations are evaluated using **true needs**. For a minimum count $m$, the DP requires
$|S\cap G_0|\ge m$ and $|S\cap G_1|\ge m$, while selecting exactly $k$ people.

- **m = 1: current LDA-exp baseline.** The need rounding, utility threshold, and tie-breaking are unchanged.
- **m = 2 or 3:** the same DP with a larger minimum selected count in each group.
- If a higher minimum cannot be met with the existing utility requirement, return the
  original LDA-exp allocation for that model and sample, and record the fallback.
- If the baseline itself is infeasible, its existing top-k fallback is preserved and explicitly flagged.

All runs use predicted needs only: α = 0, β = 1, δ = 1. The original predictions are rounded once
to whole need units. Probabilities, Rashomon membership, samples, and utility tolerance stay fixed.
The utility requirement is $\sum_{i\in S}p_i\ge\sum_{i\in\mathrm{top}\text{-}k}p_i-k\epsilon$.
OPT and the true-needs DP are not run as separate algorithms.

This is an experiment, not a guarantee of lower disparity: a count constraint narrows the set of
allocations, and its effect on true-need disparity must be measured. Infeasible cases are kept in
the overall comparison through the baseline fallback rather than dropped.


In [1]:
import hashlib
import importlib
import json
import sys
from ast import literal_eval
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/allocations.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the research repository or its test/ folder.")
sys.path.insert(0, str(ROOT))
import src.allocations
importlib.reload(src.allocations)
from src.allocations import Allocations


## Settings

Defaults match the completed prediction-error experiment: n = 100, k = 10, up to 100 candidate
models per seed, and five samples per seed. Start with minima 1, 2, and 3. Keep 1 in the list:
it supplies a freshly computed, paired baseline. Every minimum must satisfy 2m ≤ k.

For a quick pilot, reduce `N_MODELS` and `ITERS_PER_SEED`. Reducing the candidate model pool
can change Rashomon membership; pilot and full runs therefore have separate output folders.


In [2]:
METHOD = "bootstrap"
MODEL_CLASS = "nn_multiclass"
PRED_METHOD = "average"
NEED_THRESHOLD = 1
NEED_COL = "gagne_sum_t"
EXP_NEED_COL = "exp_need_{model}"
GROUP_COL = "race"
RASHOMON_EPS = 0.01
ALLOCATION_EPS = 0.1
N_MODELS = 100
SAMPLE_SIZE = 100
SELECTION_RATE = 0.1
ITERS_PER_SEED = 5

MIN_SELECTED_PER_GROUP = [1, 2, 3]
SKIP_EXISTING = True

PREDICTION_PATH = ROOT / (
    f"data/predictions/test_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_qualified_gagne_{NEED_THRESHOLD}.csv"
)
RAW_DATA_PATH = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
RESULTS_ROOT = ROOT / "data/results/allocations/minimum_group_count"


In [3]:
if not MIN_SELECTED_PER_GROUP or 1 not in MIN_SELECTED_PER_GROUP:
    raise ValueError("Include minimum 1 as the paired LDA-exp baseline.")
if len(set(MIN_SELECTED_PER_GROUP)) != len(MIN_SELECTED_PER_GROUP):
    raise ValueError("Duplicate minima would repeat a run.")
if PRED_METHOD != "average":
    raise ValueError("Use average predictions to match the existing LDA-exp experiment.")

predictions = pd.read_csv(PREDICTION_PATH)
groups = pd.read_csv(RAW_DATA_PATH, usecols=["person_id", GROUP_COL])
df = predictions.merge(groups, on="person_id", how="left", validate="many_to_one")
model_columns = [f"m_{i}" for i in range(1, N_MODELS + 1)]
need_columns = [Allocations.exp_need_col_for_model(model, EXP_NEED_COL) for model in model_columns]
required = ["person_id", "seed", "y", NEED_COL, GROUP_COL, *model_columns, *need_columns]
missing = [column for column in required if column not in df]
if missing:
    raise ValueError(f"Missing prediction columns: {missing}")
seeds = sorted(df.seed.unique().tolist())
if seeds != list(range(len(seeds))):
    raise ValueError("The cohort sampler expects consecutive seed IDs starting at zero.")
people_rows = df.person_id.ge(0)
if not df.loc[people_rows, GROUP_COL].isin([0, 1]).all():
    raise ValueError("Every person must have a group label of 0 or 1.")
if df.loc[people_rows].duplicated(["seed", "person_id"]).any():
    raise ValueError("Person IDs must be unique within each seed.")
if not np.isfinite(df.loc[people_rows, [NEED_COL, *model_columns, *need_columns]].to_numpy(dtype=float)).all():
    raise ValueError("Needs and predictions must be finite.")
if not df.loc[people_rows, "y"].isin([0, 1]).all():
    raise ValueError("Qualification labels must be binary.")

def make_allocator():
    return Allocations(
        df=df, sample_size=SAMPLE_SIZE, selection_rate=SELECTION_RATE,
        qualification_col="y", need_col=NEED_COL, group_col=GROUP_COL,
        rashomon_eps=RASHOMON_EPS, allocation_eps=ALLOCATION_EPS,
        n_models=N_MODELS, iters_per_seed=ITERS_PER_SEED,
    )

reference_allocator = make_allocator()
for minimum in MIN_SELECTED_PER_GROUP:
    reference_allocator._validate_min_selected_per_group(minimum)
minima = sorted(MIN_SELECTED_PER_GROUP)
eligible_models = {
    int(seed): reference_allocator.rashomon_models(df.loc[df.seed.eq(seed) & df.person_id.lt(0)])
    for seed in seeds
}
expected_keys = {(seed, iteration, model)
                 for seed, models in eligible_models.items()
                 for iteration in range(ITERS_PER_SEED) for model in models}
split_by_seed = {seed: df.loc[df.seed.eq(seed) & people_rows].set_index("person_id", drop=False) for seed in seeds}
print(f"n={SAMPLE_SIZE}, k={reference_allocator.k}; {len(seeds)} seeds, {ITERS_PER_SEED} samples per seed")
print("Eligible models by seed:", {seed: len(models) for seed, models in eligible_models.items()})
print(f"{len(minima)} LDA-exp runs, including the minimum-one baseline.")


n=100, k=10; 5 seeds, 5 samples per seed
Eligible models by seed: {0: 100, 1: 100, 2: 100, 3: 100, 4: 98}
3 LDA-exp runs, including the minimum-one baseline.


## Saved outputs and fallback diagnostics

Each configuration has its own directory under `data/results/allocations/minimum_group_count/`.
The configuration includes all input hashes and the allocation-code hash. Each minimum saves
allocations, sampled people, one diagnostic row per eligible model/sample, true-need evaluation
metrics, and a completion marker with output hashes and runtime. Identical allocations may be
merged in the allocation file, but diagnostics and metrics retain one row per model/sample.

`constraint_feasible` says whether the requested minimum can meet the utility requirement.
`fallback_used` is its complement. `fallback_reason` distinguishes insufficient group members from
an unattainable utility requirement. `selection_source` identifies `constrained_dp`, `baseline_dp`,
or the baseline's last-resort `top_k`. The selected and available counts for each group are also saved.

Completed runs are checked before reuse. Cohort identities and model coverage must match across
all minima. Every higher-minimum fallback is checked against the corresponding baseline selection.


In [4]:
def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

config = {
    "schema_version": 1, "experiment": "minimum_group_count",
    "alpha": 0.0, "beta": 1.0, "need_step": 1.0, "shrinkage_order": "after_rounding",
    "fallback_policy": "original_lda_exp_including_its_top_k_fallback",
    "method": METHOD, "model_class": MODEL_CLASS, "prediction_method": PRED_METHOD,
    "need_threshold": NEED_THRESHOLD, "need_column": NEED_COL,
    "expected_need_column": EXP_NEED_COL, "group_column": GROUP_COL,
    "rashomon_epsilon": RASHOMON_EPS, "allocation_epsilon": ALLOCATION_EPS,
    "n_models": N_MODELS, "sample_size": SAMPLE_SIZE, "selection_rate": SELECTION_RATE,
    "k": reference_allocator.k, "iters_per_seed": ITERS_PER_SEED, "seeds": seeds,
    "prediction_path": str(PREDICTION_PATH), "prediction_sha256": file_hash(PREDICTION_PATH),
    "raw_data_path": str(RAW_DATA_PATH), "raw_data_sha256": file_hash(RAW_DATA_PATH),
    "allocation_code_sha256": file_hash(ROOT / "src/allocations.py"),
}
config_text = json.dumps(config, sort_keys=True, indent=2)
config_id = hashlib.sha256(config_text.encode()).hexdigest()[:12]
OUTPUT_DIR = RESULTS_ROOT / f"size-{SAMPLE_SIZE}_models-{N_MODELS}_iters-{ITERS_PER_SEED}_{config_id}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / "config.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != config:
        raise ValueError("This output directory contains a different configuration.")
else:
    manifest_path.write_text(config_text + "\n")

def algorithm_name(minimum):
    return "LDA-exp" if minimum == 1 else f"LDA-exp-min_group_{minimum}"

def run_paths(minimum):
    return {name: OUTPUT_DIR / f"{name}_min_group-{minimum}.csv"
            for name in ("allocations", "people", "diagnostics", "metrics")} | {
        "metadata": OUTPUT_DIR / f"run_min_group-{minimum}.json"}

def cohort_identity(people):
    expected = {(seed, i) for seed in seeds for i in range(ITERS_PER_SEED)}
    if set(zip(people.seed, people.iteration)) != expected or len(people) != len(expected):
        raise ValueError("Unexpected sample coverage.")
    identity = []
    for row in people.sort_values(["seed", "iteration"]).itertuples(index=False):
        if len(row.people) != SAMPLE_SIZE or len(set(row.people)) != SAMPLE_SIZE:
            raise ValueError("A sample contains missing or duplicate people.")
        if not set(row.people) <= set(split_by_seed[row.seed].index):
            raise ValueError("A sample includes a person outside its prediction split.")
        identity.append((int(row.seed), int(row.iteration), tuple(row.people)))
    return tuple(identity)

print("Results directory:", OUTPUT_DIR)


Results directory: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/results/allocations/minimum_group_count/size-100_models-100_iters-5_bcb17c521c48


In [5]:
def validate_and_evaluate(allocations, people, diagnostics, minimum, baseline=None):
    identity = cohort_identity(people)
    keys = list(zip(diagnostics.seed, diagnostics.iteration, diagnostics.model))
    if set(keys) != expected_keys or len(keys) != len(expected_keys):
        raise ValueError("Diagnostics do not cover each eligible model/sample exactly once.")
    if not diagnostics.min_selected_per_group.eq(minimum).all():
        raise ValueError("Saved diagnostics have the wrong minimum selected count.")
    for col in ("constraint_feasible", "fallback_used", "baseline_feasible"):
        if not diagnostics[col].isin([True, False]).all():
            raise ValueError(f"Invalid boolean diagnostic: {col}")
    diag_by_key = diagnostics.set_index(["seed", "iteration", "model"])
    cohorts = {(seed, iteration): split_by_seed[seed].loc[list(ids)] for seed, iteration, ids in identity}
    selections, records = {}, []

    for row in allocations.itertuples(index=False):
        cohort = cohorts[(row.seed, row.iteration)]
        selected = tuple(sorted(row.selected))
        if len(selected) != config["k"] or len(set(selected)) != config["k"] or not set(selected) <= set(cohort.index):
            raise ValueError("An allocation has the wrong size or contains invalid people.")
        chosen = cohort.loc[list(selected)]
        counts = [int(chosen[GROUP_COL].eq(g).sum()) for g in (0, 1)]
        available = [int(cohort[GROUP_COL].eq(g).sum()) for g in (0, 1)]
        true_gap = (abs(chosen.loc[chosen[GROUP_COL].eq(1), NEED_COL].mean()
                        - chosen.loc[chosen[GROUP_COL].eq(0), NEED_COL].mean()) if min(counts) else np.nan)
        for model in row.models:
            key = (row.seed, row.iteration, model)
            if key not in expected_keys or key in selections:
                raise ValueError("Allocation model coverage is duplicated or unexpected.")
            selections[key] = selected
            diagnostic = diag_by_key.loc[key]
            if counts != [diagnostic.selected_count_0, diagnostic.selected_count_1]:
                raise ValueError("Diagnostic selected counts disagree with the allocation.")
            if available != [diagnostic.available_count_0, diagnostic.available_count_1]:
                raise ValueError("Diagnostic group availability disagrees with the sample.")
            if bool(diagnostic.constraint_feasible) == bool(diagnostic.fallback_used):
                raise ValueError("Feasibility and fallback flags disagree.")
            if diagnostic.constraint_feasible and min(counts) < minimum:
                raise ValueError("A feasible allocation violates the requested group minimum.")
            if minimum > 1 and diagnostic.fallback_used and selected != baseline[key]:
                raise ValueError("A fallback differs from the paired LDA-exp baseline.")
            utility_sum = float(chosen[model].sum())
            utility_floor = float(cohort[model].nlargest(config["k"]).sum() - config["k"] * ALLOCATION_EPS)
            if not np.isclose(utility_sum, diagnostic.utility_sum, rtol=0, atol=1e-10):
                raise ValueError("Diagnostic utility disagrees with the allocation.")
            if not np.isclose(utility_floor, diagnostic.utility_requirement, rtol=0, atol=1e-10):
                raise ValueError("The utility requirement changed.")
            if utility_sum < utility_floor - 1e-10:
                raise ValueError("An allocation violates the utility requirement.")
            if not diagnostic.fallback_used and diagnostic.fallback_reason != "none":
                raise ValueError("A feasible allocation has a fallback reason.")
            if diagnostic.selection_source == "top_k" and diagnostic.baseline_feasible:
                raise ValueError("The top-k fallback is incorrectly marked baseline-feasible.")
            predicted = np.rint(chosen[Allocations.exp_need_col_for_model(model, EXP_NEED_COL)])
            predicted_gap = (abs(predicted.loc[chosen[GROUP_COL].eq(1)].mean()
                                 - predicted.loc[chosen[GROUP_COL].eq(0)].mean()) if min(counts) else np.nan)
            records.append({
                "seed": row.seed, "iteration": row.iteration, "model": model,
                "algorithm": algorithm_name(minimum), "selected": list(selected),
                **diagnostic.to_dict(), "minimum_selected_count": min(counts),
                "absolute_disparity": true_gap, "predicted_disparity": predicted_gap,
                "expected_utility": utility_sum / config["k"],
                "realized_utility": float(chosen.y.mean()),
            })
    if set(selections) != expected_keys:
        raise ValueError("An eligible model/sample allocation is missing.")
    return pd.DataFrame(records), selections


In [6]:
run_records, metric_frames = [], []
reference_cohorts, baseline_selections = None, None

for minimum in minima:
    paths = run_paths(minimum)
    if SKIP_EXISTING and paths["metadata"].exists():
        metadata = json.loads(paths["metadata"].read_text())
        if metadata["config_id"] != config_id or metadata["min_selected_per_group"] != minimum:
            raise ValueError(f"Mismatched metadata: {paths['metadata']}")
        for name in ("allocations", "people", "diagnostics", "metrics"):
            if not paths[name].exists() or file_hash(paths[name]) != metadata["sha256"][name]:
                raise ValueError(f"Incomplete or changed saved result: {paths[name]}")
        allocations = pd.read_csv(paths["allocations"], converters={"selected": literal_eval, "models": literal_eval})
        people = pd.read_csv(paths["people"], converters={"people": literal_eval})
        diagnostics = pd.read_csv(paths["diagnostics"])
        print(f"Using saved {algorithm_name(minimum)}")
        save_run = False
    else:
        print(f"Running {algorithm_name(minimum)}")
        allocator = make_allocator()
        started = perf_counter()
        opt, true_dp, allocations, people = allocator.get_allocations(
            exp_need_col=EXP_NEED_COL, algorithms="LDA-exp",
            alpha=0.0, beta=1.0, shrinkage_order="after_rounding", need_step=1.0,
            min_selected_per_group=minimum,
        )
        elapsed = perf_counter() - started
        if not opt.empty or not true_dp.empty:
            raise AssertionError("Only the requested LDA-exp variant should run.")
        diagnostics = pd.DataFrame(allocator.group_count_diagnostics)
        metadata = {
            "config_id": config_id, "algorithm": algorithm_name(minimum),
            "min_selected_per_group": int(minimum), "elapsed_seconds": elapsed,
            "n_cohorts": len(people), "n_model_cohort_allocations": len(diagnostics),
            "n_saved_allocation_rows": len(allocations),
        }
        save_run = True

    identity = cohort_identity(people)
    if reference_cohorts is None:
        reference_cohorts = identity
    elif identity != reference_cohorts:
        raise ValueError("Sample identities or row order changed between minima.")
    metrics, selections = validate_and_evaluate(allocations, people, diagnostics, minimum, baseline_selections)
    if minimum == 1:
        baseline_selections = selections

    if save_run:
        paths["metadata"].unlink(missing_ok=True)
        for name, frame in {"allocations": allocations, "people": people, "diagnostics": diagnostics, "metrics": metrics}.items():
            temporary = paths[name].with_suffix(".csv.tmp")
            frame.to_csv(temporary, index=False)
            temporary.replace(paths[name])
        metadata["n_fallbacks"] = int(metrics.fallback_used.sum())
        metadata["sha256"] = {name: file_hash(paths[name]) for name in ("allocations", "people", "diagnostics", "metrics")}
        temporary = paths["metadata"].with_suffix(".json.tmp")
        temporary.write_text(json.dumps(metadata, indent=2) + "\n")
        temporary.replace(paths["metadata"])
        print(f"Saved in {metadata['elapsed_seconds']:.1f} seconds; {metadata['n_fallbacks']} fallbacks")

    run_records.append({key: value for key, value in metadata.items() if key != "sha256"})
    metric_frames.append(metrics)

run_summary = pd.DataFrame(run_records).sort_values("min_selected_per_group")
all_metrics = pd.concat(metric_frames, ignore_index=True)
run_summary.to_csv(OUTPUT_DIR / "run_summary.csv", index=False)
all_metrics.to_csv(OUTPUT_DIR / "allocation_metrics.csv", index=False)
display(run_summary[["algorithm", "elapsed_seconds", "n_cohorts", "n_model_cohort_allocations", "n_fallbacks"]])


Running LDA-exp


[Seed 4]: 100%|██████████| 5/5 [00:34<00:00,  6.80s/it]


Saved in 154.4 seconds; 0 fallbacks
Running LDA-exp-min_group_2


[Seed 4]: 100%|██████████| 5/5 [00:34<00:00,  6.92s/it]


Saved in 156.4 seconds; 298 fallbacks
Running LDA-exp-min_group_3


[Seed 4]: 100%|██████████| 5/5 [00:33<00:00,  6.76s/it]


Saved in 151.5 seconds; 389 fallbacks


,algorithm,elapsed_seconds,n_cohorts,n_model_cohort_allocations,n_fallbacks
0,LDA-exp,154.449191,25,2490,0
1,LDA-exp-min_group_2,156.372915,25,2490,298
2,LDA-exp-min_group_3,151.537928,25,2490,389


## Initial comparison

The overall summaries include fallback allocations. Disparity is undefined when only one group
is selected; such observations remain in the saved data, are counted explicitly, and are excluded
only from disparity means. Standard deviations describe variation across model/sample allocations,
not confidence intervals. These observations share models and samples and are not independent.

The paired change is variant disparity minus baseline disparity on the same model/sample; negative
means improvement. Compare the mean and SD alongside the fallback rate and utility. A high fallback
rate means the stronger constraint often cannot be applied. Do not compare only the feasible subset
against the entire baseline population.


In [8]:
summary = all_metrics.groupby(["min_selected_per_group", "algorithm"]).agg(
    n_total=("absolute_disparity", "size"), n_valid=("absolute_disparity", "count"),
    absolute_disparity_mean=("absolute_disparity", "mean"),
    absolute_disparity_std=("absolute_disparity", "std"),
    expected_utility=("expected_utility", "mean"), realized_utility=("realized_utility", "mean"),
    constraint_feasible_rate=("constraint_feasible", "mean"), fallback_rate=("fallback_used", "mean"),
    mean_minimum_selected_count=("minimum_selected_count", "mean"),
).reset_index()
summary["n_undefined"] = summary.n_total - summary.n_valid
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
display(summary.drop(columns=["n_total", "n_valid"]))

baseline_metrics = all_metrics.loc[all_metrics.min_selected_per_group.eq(1),
                                   ["seed", "iteration", "model", "absolute_disparity"]].rename(
    columns={"absolute_disparity": "baseline_disparity"})
paired = all_metrics.merge(baseline_metrics, on=["seed", "iteration", "model"], validate="many_to_one")
paired["disparity_change"] = paired.absolute_disparity - paired.baseline_disparity
paired_summary = paired.groupby(["min_selected_per_group", "algorithm"]).agg(
    n_valid_pairs=("disparity_change", "count"),
    mean_disparity_change=("disparity_change", "mean"),
    std_disparity_change=("disparity_change", "std"),
).reset_index()
paired_summary.to_csv(OUTPUT_DIR / "paired_summary.csv", index=False)
paired.to_csv(OUTPUT_DIR / "paired_allocation_metrics.csv", index=False)
fallback_summary = all_metrics.groupby(["min_selected_per_group", "algorithm", "fallback_reason", "selection_source"]).size().rename("n_allocations").reset_index()
fallback_summary.to_csv(OUTPUT_DIR / "fallback_summary.csv", index=False)
display(paired_summary)
display(fallback_summary)
print("Saved allocations, diagnostics, evaluations, and summaries to:", OUTPUT_DIR)


,min_selected_per_group,algorithm,absolute_disparity_mean,absolute_disparity_std,expected_utility,realized_utility,constraint_feasible_rate,fallback_rate,mean_minimum_selected_count,n_undefined
0,1,LDA-exp,1.650181,1.151700,0.972401,0.963896,1.000000,0.000000,2.642972,0
1,2,LDA-exp-min_group_2,1.521183,1.087351,0.971068,0.963695,0.880321,0.119679,2.880321,0
2,3,LDA-exp-min_group_3,1.459753,1.081591,0.968637,0.960161,0.843775,0.156225,3.404418,0


,min_selected_per_group,algorithm,n_valid_pairs,mean_disparity_change,std_disparity_change
0,1,LDA-exp,2490,0.000000,0.000000
1,2,LDA-exp-min_group_2,2490,-0.128999,0.473824
2,3,LDA-exp-min_group_3,2490,-0.190428,0.704271


,min_selected_per_group,algorithm,fallback_reason,selection_source,n_allocations
0,1,LDA-exp,none,baseline_dp,2490
1,2,LDA-exp-min_group_2,insufficient_group_members,baseline_dp,98
2,2,LDA-exp-min_group_2,none,constrained_dp,2192
3,2,LDA-exp-min_group_2,utility_requirement,baseline_dp,200
4,3,LDA-exp-min_group_3,insufficient_group_members,baseline_dp,198
5,3,LDA-exp-min_group_3,none,constrained_dp,2101
6,3,LDA-exp-min_group_3,utility_requirement,baseline_dp,191


Saved allocations, diagnostics, evaluations, and summaries to: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/results/allocations/minimum_group_count/size-100_models-100_iters-5_bcb17c521c48
